In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

PROJECT_NAME = "Personal Knowledge Decay Predictor"

matches = [
    p for p in Path("/content/drive/MyDrive").rglob(PROJECT_NAME)
    if p.is_dir()
]

PROJECT_ROOT = matches[0]

sys.path.append(str(PROJECT_ROOT))

from config import *

print("✅ Project Root :", PROJECT_ROOT)

✅ Project Root : /content/drive/MyDrive/Marwadi_uiniversity/Personal Knowledge Decay Predictor
✅ Project Root : /content/drive/MyDrive/Marwadi_uiniversity/Personal Knowledge Decay Predictor


# 1. Import Libraries

## Objective

Import the libraries required for model inference.

## Methodology

The notebook loads the trained machine learning model and performs predictions on unseen student records.

## Expected Output

All required libraries are imported successfully.

In [3]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import json
import joblib
import numpy as np
import pandas as pd

# 2. Load Saved Model

## Objective

Load the trained Logistic Regression model developed in Notebook 05.

## Expected Output

The trained prediction model.

In [4]:
best_model = joblib.load(
    f"{MODEL_DIR}/best_model.pkl"
)

print("Model Type:", type(best_model))

Model Type: <class 'sklearn.linear_model._logistic.LogisticRegression'>


📒 Section 3 — Load Scaler

In [5]:
scaler = joblib.load(
    MODEL_DIR / "scaler.pkl"
)

print("Scaler Loaded")

Scaler Loaded


📒 Section 4 — Load Feature List

In [6]:
with open(
    MODEL_DIR / "selected_features.json",
    "r"
) as f:

    feature_columns = json.load(f)

feature_columns

['interaction_order',
 'past_attempts',
 'past_correct',
 'past_accuracy',
 'rolling_accuracy',
 'mastered']

📒 Section 5 — Prediction Function

In [7]:
def predict_forgetting(student_features):
    """
    Predict whether a student is likely to experience
    a forgetting event.

    Parameters
    ----------
    student_features : list

    Returns
    -------
    Prediction
    Probability
    """

    student_df = pd.DataFrame(
        [student_features],
        columns=feature_columns
    )

    student_scaled = scaler.transform(student_df)

    prediction = best_model.predict(student_scaled)[0]

    probability = best_model.predict_proba(student_scaled)[0][1]

    return prediction, probability

📒 Section 6 — Predict for One Student

In [8]:
sample_student = [
    12,      # interaction_order
    12,      # past_attempts
    6,       # past_correct
    0.50,    # past_accuracy
    0.40,    # rolling_accuracy
    1        # mastered
]

prediction, probability = predict_forgetting(sample_student)

print("Prediction :", prediction)

print("Probability:", round(probability,4))

Prediction : 1
Probability: 0.705


📒 Section 7 — Human-Readable Interpretation

In [9]:
if prediction == 1:

    print("⚠️ High Risk of Forgetting")

else:

    print("✅ Low Risk of Forgetting")

print(f"Probability : {probability:.2%}")

⚠️ High Risk of Forgetting
Probability : 70.50%


📒 Section 8 — Batch Prediction

In [10]:
sample_students = pd.DataFrame([
    [10,10,7,0.70,0.80,1],
    [15,15,5,0.30,0.20,0],
    [8,8,6,0.75,0.80,1],
    [20,20,8,0.40,0.20,1]
], columns=feature_columns)

scaled = scaler.transform(sample_students)

sample_students["Prediction"] = best_model.predict(scaled)

sample_students["Probability"] = best_model.predict_proba(scaled)[:,1]

display(sample_students)

,interaction_order,past_attempts,past_correct,past_accuracy,rolling_accuracy,mastered,Prediction,Probability
0,10,10,7,0.70,0.8,1,1,0.505827
1,15,15,5,0.30,0.2,0,0,0.000008
2,8,8,6,0.75,0.8,1,0,0.485437
3,20,20,8,0.40,0.2,1,1,0.793986


# Prediction Interpretation

The trained model estimates the probability that a learner will experience a forgetting event.

Predictions can be integrated into an adaptive learning platform to:

- identify learners at risk of forgetting,
- recommend timely revision,
- prioritize instructional interventions, and
- support personalized learning pathways.

# Notebook Summary

This notebook demonstrated the deployment-ready prediction pipeline.

The trained Logistic Regression model was successfully loaded and used to predict forgetting events for new student records.

Both single-student and batch prediction workflows were demonstrated.

This notebook represents the inference component of the knowledge decay prediction system.